# BRForecast — Fase 6: Analise de Odds

Compara as probabilidades do modelo (Poisson/Dixon-Coles + ELO) com as odds das casas de apostas.

**Secoes:**
1. Setup e carga de dados
2. Conversao de odds para probabilidades implicitas
3. Probabilidades do modelo por jogo
4. Tabela comparativa (modelo vs odds)
5. Metricas globais (Brier Score, Log-Loss)
6. Grafico de calibracao
7. Mapa de divergencias (top 20)

## 1. Setup e carga de dados

In [ ]:
import sys
import os

# Adicionar raiz do projeto ao path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.config import (
    TARGET_YEAR, HFA, ELO_LAMBDA_WEIGHT, DIXON_COLES_RHO,
    MIN_MATCHES_SEASON, SERIE_A_IDS, ELO_WINDOW_START,
)
from src.elo import load_historical_matches, calculate_all_elos, expected_score
from src.poisson import (
    load_serie_a_with_xg, calculate_team_strengths,
    calculate_lambdas, score_probabilities,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup OK')

In [ ]:
# --- Carregar ELO ---
elo_matches = load_historical_matches()
elo_ratings, elo_history = calculate_all_elos(elo_matches)
print(f'{len(elo_ratings)} times com ELO')

# --- Carregar partidas com xG e odds ---
matches_xg = load_serie_a_with_xg()
completed = matches_xg[matches_xg['status'] == 'complete'].copy()
print(f'{len(completed)} partidas completas (Serie A {ELO_WINDOW_START}-{TARGET_YEAR})')

# --- Forcas Poisson (temporada alvo + fallback) ---
target_m = matches_xg[matches_xg['season_year'] == TARGET_YEAR]
target_done = target_m[target_m['status'] == 'complete']

if len(target_done) < MIN_MATCHES_SEASON:
    print(f'Apenas {len(target_done)} jogos em {TARGET_YEAR}, incluindo {TARGET_YEAR - 1}')
    expanded = matches_xg[matches_xg['season_year'] >= TARGET_YEAR - 1]
    team_strengths, league_avgs = calculate_team_strengths(expanded)
else:
    team_strengths, league_avgs = calculate_team_strengths(target_done)

print(f'{len(team_strengths)} times com forcas ({league_avgs["metric"]})')
print(f'Media liga: mandante={league_avgs["avg_home_goals"]:.3f}, visitante={league_avgs["avg_away_goals"]:.3f}')

In [ ]:
# Filtrar partidas com odds validas
completed['odds_ft_1'] = pd.to_numeric(completed['odds_ft_1'], errors='coerce')
completed['odds_ft_x'] = pd.to_numeric(completed['odds_ft_x'], errors='coerce')
completed['odds_ft_2'] = pd.to_numeric(completed['odds_ft_2'], errors='coerce')

has_odds = completed.dropna(subset=['odds_ft_1', 'odds_ft_x', 'odds_ft_2']).copy()
has_odds = has_odds[(has_odds['odds_ft_1'] > 1.0) &
                    (has_odds['odds_ft_x'] > 1.0) &
                    (has_odds['odds_ft_2'] > 1.0)]

print(f'{len(has_odds)} partidas com odds validas de {len(completed)} completas '
      f'({100*len(has_odds)/len(completed):.1f}%)')
print(f'Temporadas: {sorted(has_odds["season_year"].unique())}')

## 2. Conversao de odds para probabilidades implicitas

In [ ]:
def odds_to_probs(odds_h, odds_d, odds_a):
    """Converte odds decimais para probabilidades normalizadas (remove overround).
    
    P_implied(x) = 1 / odds(x)
    P_normalized(x) = P_implied(x) / sum(P_implied)
    """
    p_h = 1.0 / odds_h
    p_d = 1.0 / odds_d
    p_a = 1.0 / odds_a
    total = p_h + p_d + p_a  # > 1.0 por causa do overround (margem da casa)
    return p_h / total, p_d / total, p_a / total


# Aplicar conversao
odds_probs = has_odds.apply(
    lambda r: odds_to_probs(r['odds_ft_1'], r['odds_ft_x'], r['odds_ft_2']),
    axis=1, result_type='expand'
)
has_odds['odds_p_home'] = odds_probs[0]
has_odds['odds_p_draw'] = odds_probs[1]
has_odds['odds_p_away'] = odds_probs[2]

# Overround medio
has_odds['overround'] = (1.0/has_odds['odds_ft_1'] + 
                         1.0/has_odds['odds_ft_x'] + 
                         1.0/has_odds['odds_ft_2'])

print(f'Overround medio: {has_odds["overround"].mean():.3f} '
      f'(margem ~{(has_odds["overround"].mean() - 1)*100:.1f}%)')
print(f'Overround min: {has_odds["overround"].min():.3f}, '
      f'max: {has_odds["overround"].max():.3f}')

# Verificacao: probabilidades somam ~1
check = has_odds['odds_p_home'] + has_odds['odds_p_draw'] + has_odds['odds_p_away']
print(f'Soma probs normalizadas: min={check.min():.6f}, max={check.max():.6f}')

has_odds[['home_name', 'away_name', 'odds_ft_1', 'odds_ft_x', 'odds_ft_2',
          'odds_p_home', 'odds_p_draw', 'odds_p_away', 'overround']].head(10)

## 3. Probabilidades do modelo por jogo

In [ ]:
# Calcular probabilidades do modelo (Dixon-Coles + ELO) para cada jogo
model_probs = []

for idx, match in has_odds.iterrows():
    home = match['home_name']
    away = match['away_name']
    
    lam_h, lam_a = calculate_lambdas(
        home, away, team_strengths, league_avgs,
        elo_ratings=elo_ratings,
    )
    
    probs = score_probabilities(lam_h, lam_a)
    
    # Resultado real
    hg = match['homeGoalCount']
    ag = match['awayGoalCount']
    if hg > ag:
        result = 'H'
    elif hg == ag:
        result = 'D'
    else:
        result = 'A'
    
    model_probs.append({
        'idx': idx,
        'model_p_home': probs['home_win'],
        'model_p_draw': probs['draw'],
        'model_p_away': probs['away_win'],
        'lambda_home': lam_h,
        'lambda_away': lam_a,
        'result': result,
    })

model_df = pd.DataFrame(model_probs).set_index('idx')
has_odds = has_odds.join(model_df)

# Verificacao: probabilidades do modelo somam ~1
check_m = has_odds['model_p_home'] + has_odds['model_p_draw'] + has_odds['model_p_away']
print(f'Probs modelo: min={check_m.min():.6f}, max={check_m.max():.6f}')
print(f'{len(has_odds)} jogos com modelo + odds prontos para comparacao')

## 4. Tabela comparativa (modelo vs odds)

In [ ]:
# Montar tabela legivel
comparison = has_odds[[
    'home_name', 'away_name', 'season_year', 'game_week',
    'homeGoalCount', 'awayGoalCount', 'result',
    'model_p_home', 'model_p_draw', 'model_p_away',
    'odds_p_home', 'odds_p_draw', 'odds_p_away',
]].copy()

comparison.columns = [
    'Mandante', 'Visitante', 'Ano', 'Rodada',
    'Gols_H', 'Gols_A', 'Resultado',
    'Mod_H', 'Mod_D', 'Mod_A',
    'Odds_H', 'Odds_D', 'Odds_A',
]

print(f'Tabela comparativa: {len(comparison)} jogos')
print(f'Temporadas: {sorted(comparison["Ano"].unique())}')
comparison.head(20)

## 5. Metricas globais (Brier Score e Log-Loss)

In [ ]:
def compute_metrics(df, p_home_col, p_draw_col, p_away_col, label):
    """Calcula Brier Score e Log-Loss para um conjunto de previsoes.
    
    Brier Score: mean((P_pred - resultado)^2) — multiclasse
    Log-Loss: mean(-log(P_resultado_correto))
    """
    brier_sum = 0.0
    logloss_sum = 0.0
    n = 0
    
    for _, row in df.iterrows():
        p_h = row[p_home_col]
        p_d = row[p_draw_col]
        p_a = row[p_away_col]
        
        hg = row['homeGoalCount']
        ag = row['awayGoalCount']
        if hg > ag:
            actual = (1, 0, 0)
        elif hg == ag:
            actual = (0, 1, 0)
        else:
            actual = (0, 0, 1)
        
        # Brier Score multiclasse
        brier_sum += (p_h - actual[0])**2 + (p_d - actual[1])**2 + (p_a - actual[2])**2
        
        # Log-Loss
        p_actual = p_h * actual[0] + p_d * actual[1] + p_a * actual[2]
        logloss_sum += -np.log(max(p_actual, 1e-10))
        
        n += 1
    
    return {
        'label': label,
        'n_matches': n,
        'brier_score': brier_sum / n,
        'log_loss': logloss_sum / n,
    }


# --- Metricas globais (todas as temporadas) ---
m_model = compute_metrics(has_odds, 'model_p_home', 'model_p_draw', 'model_p_away', 'Modelo (DC+ELO)')
m_odds = compute_metrics(has_odds, 'odds_p_home', 'odds_p_draw', 'odds_p_away', 'Odds (casas)')

metrics_all = pd.DataFrame([m_model, m_odds])
print('=== Metricas Globais (todas as temporadas) ===')
print(metrics_all.to_string(index=False))

# --- Metricas por temporada ---
print('\n=== Metricas por temporada ===')
for year in sorted(has_odds['season_year'].unique()):
    subset = has_odds[has_odds['season_year'] == year]
    if len(subset) < 20:
        continue
    m_mod_y = compute_metrics(subset, 'model_p_home', 'model_p_draw', 'model_p_away', f'Modelo {year}')
    m_odd_y = compute_metrics(subset, 'odds_p_home', 'odds_p_draw', 'odds_p_away', f'Odds {year}')
    print(f'  {year}: Modelo BS={m_mod_y["brier_score"]:.5f} LL={m_mod_y["log_loss"]:.4f}  |  '
          f'Odds BS={m_odd_y["brier_score"]:.5f} LL={m_odd_y["log_loss"]:.4f}  |  '
          f'Diff BS={m_mod_y["brier_score"] - m_odd_y["brier_score"]:+.5f}  '
          f'({len(subset)} jogos)')

In [ ]:
# Grafico comparativo Brier Score (modelo vs odds)
years = sorted(has_odds['season_year'].unique())
bs_model = []
bs_odds = []
valid_years = []

for year in years:
    subset = has_odds[has_odds['season_year'] == year]
    if len(subset) < 20:
        continue
    valid_years.append(year)
    bs_model.append(compute_metrics(subset, 'model_p_home', 'model_p_draw', 'model_p_away', '')['brier_score'])
    bs_odds.append(compute_metrics(subset, 'odds_p_home', 'odds_p_draw', 'odds_p_away', '')['brier_score'])

fig_bs = go.Figure()
fig_bs.add_trace(go.Bar(x=[str(y) for y in valid_years], y=bs_model, name='Modelo (DC+ELO)',
                        marker_color='#636EFA'))
fig_bs.add_trace(go.Bar(x=[str(y) for y in valid_years], y=bs_odds, name='Odds (casas)',
                        marker_color='#EF553B'))
fig_bs.update_layout(
    title='Brier Score por temporada: Modelo vs Odds',
    xaxis_title='Temporada',
    yaxis_title='Brier Score (menor = melhor)',
    barmode='group',
    template='plotly_white',
    height=400,
)
fig_bs.show()

## 6. Grafico de calibracao

In [ ]:
def build_calibration_data(df, p_col, outcome_col_fn, label, n_bins=10):
    """Constroi dados de calibracao: agrupa previsoes em faixas e calcula taxa real.
    
    Args:
        df: DataFrame com jogos
        p_col: coluna com a probabilidade prevista
        outcome_col_fn: funcao que recebe uma row e retorna 1 se evento ocorreu, 0 senao
        label: nome para a serie
        n_bins: numero de faixas (default 10 = 0-10%, 10-20%, ...)
    
    Returns:
        DataFrame [bin_center, predicted_mean, actual_mean, count, label]
    """
    data = df.copy()
    data['_pred'] = data[p_col]
    data['_actual'] = data.apply(outcome_col_fn, axis=1)
    
    bins = np.linspace(0, 1, n_bins + 1)
    data['_bin'] = pd.cut(data['_pred'], bins=bins, include_lowest=True)
    
    cal = data.groupby('_bin', observed=True).agg(
        predicted_mean=('_pred', 'mean'),
        actual_mean=('_actual', 'mean'),
        count=('_actual', 'count'),
    ).reset_index()
    cal['label'] = label
    
    return cal


# Funcoes de outcome para cada tipo de resultado
def is_home_win(row):
    return 1 if row['homeGoalCount'] > row['awayGoalCount'] else 0

def is_draw(row):
    return 1 if row['homeGoalCount'] == row['awayGoalCount'] else 0

def is_away_win(row):
    return 1 if row['homeGoalCount'] < row['awayGoalCount'] else 0


# Calibracao para vitoria do mandante (modelo vs odds)
cal_model_h = build_calibration_data(has_odds, 'model_p_home', is_home_win, 'Modelo - P(H)')
cal_odds_h = build_calibration_data(has_odds, 'odds_p_home', is_home_win, 'Odds - P(H)')

# Calibracao para empate
cal_model_d = build_calibration_data(has_odds, 'model_p_draw', is_draw, 'Modelo - P(D)')
cal_odds_d = build_calibration_data(has_odds, 'odds_p_draw', is_draw, 'Odds - P(D)')

# Calibracao para vitoria visitante
cal_model_a = build_calibration_data(has_odds, 'model_p_away', is_away_win, 'Modelo - P(A)')
cal_odds_a = build_calibration_data(has_odds, 'odds_p_away', is_away_win, 'Odds - P(A)')

print('Dados de calibracao calculados.')
print(f'Bins com dados (modelo P(H)): {len(cal_model_h)}')

In [ ]:
# Grafico de calibracao combinado (3 subplots: H, D, A)
fig_cal = make_subplots(
    rows=1, cols=3,
    subplot_titles=['P(Mandante)', 'P(Empate)', 'P(Visitante)'],
    horizontal_spacing=0.08,
)

# Cores
c_model = '#636EFA'
c_odds = '#EF553B'

for col_idx, (cal_mod, cal_odd, title) in enumerate([
    (cal_model_h, cal_odds_h, 'P(H)'),
    (cal_model_d, cal_odds_d, 'P(D)'),
    (cal_model_a, cal_odds_a, 'P(A)'),
], start=1):
    # Diagonal perfeita
    fig_cal.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   line=dict(color='gray', dash='dash', width=1),
                   showlegend=(col_idx == 1), name='Calibracao perfeita'),
        row=1, col=col_idx,
    )
    
    # Modelo
    if len(cal_mod) > 0:
        fig_cal.add_trace(
            go.Scatter(
                x=cal_mod['predicted_mean'], y=cal_mod['actual_mean'],
                mode='markers+lines', name='Modelo',
                marker=dict(size=cal_mod['count'].clip(upper=100) / 5 + 4, color=c_model),
                line=dict(color=c_model),
                text=[f'n={c}' for c in cal_mod['count']],
                hovertemplate='Previsto: %{x:.2f}<br>Real: %{y:.2f}<br>%{text}',
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
    
    # Odds
    if len(cal_odd) > 0:
        fig_cal.add_trace(
            go.Scatter(
                x=cal_odd['predicted_mean'], y=cal_odd['actual_mean'],
                mode='markers+lines', name='Odds',
                marker=dict(size=cal_odd['count'].clip(upper=100) / 5 + 4, color=c_odds),
                line=dict(color=c_odds),
                text=[f'n={c}' for c in cal_odd['count']],
                hovertemplate='Previsto: %{x:.2f}<br>Real: %{y:.2f}<br>%{text}',
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig_cal.update_xaxes(title_text='Probabilidade prevista', range=[0, 1])
fig_cal.update_yaxes(title_text='Frequencia real', range=[0, 1])
fig_cal.update_layout(
    title='Grafico de Calibracao: Modelo vs Odds',
    template='plotly_white',
    height=450,
    width=1100,
)
fig_cal.show()

In [ ]:
# Calibracao combinada (todas as previsoes juntas: H, D, A empilhadas)
# Util para ver visao geral da calibracao

all_preds_model = pd.concat([
    has_odds[['model_p_home', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['model_p_home'],
        actual=has_odds.apply(is_home_win, axis=1),
    ),
    has_odds[['model_p_draw', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['model_p_draw'],
        actual=has_odds.apply(is_draw, axis=1),
    ),
    has_odds[['model_p_away', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['model_p_away'],
        actual=has_odds.apply(is_away_win, axis=1),
    ),
])

all_preds_odds = pd.concat([
    has_odds[['odds_p_home', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['odds_p_home'],
        actual=has_odds.apply(is_home_win, axis=1),
    ),
    has_odds[['odds_p_draw', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['odds_p_draw'],
        actual=has_odds.apply(is_draw, axis=1),
    ),
    has_odds[['odds_p_away', 'homeGoalCount', 'awayGoalCount']].assign(
        pred=has_odds['odds_p_away'],
        actual=has_odds.apply(is_away_win, axis=1),
    ),
])

n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)

def bin_calibration(df, label):
    df = df.copy()
    df['bin'] = pd.cut(df['pred'], bins=bins, include_lowest=True)
    grouped = df.groupby('bin', observed=True).agg(
        pred_mean=('pred', 'mean'),
        actual_mean=('actual', 'mean'),
        count=('actual', 'count'),
    ).reset_index()
    grouped['label'] = label
    return grouped

cal_all_model = bin_calibration(all_preds_model, 'Modelo')
cal_all_odds = bin_calibration(all_preds_odds, 'Odds')

fig_cal_all = go.Figure()
fig_cal_all.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                                 line=dict(color='gray', dash='dash'), name='Perfeito'))

fig_cal_all.add_trace(go.Scatter(
    x=cal_all_model['pred_mean'], y=cal_all_model['actual_mean'],
    mode='markers+lines', name='Modelo',
    marker=dict(size=10, color=c_model), line=dict(color=c_model),
    text=[f'n={c}' for c in cal_all_model['count']],
    hovertemplate='Previsto: %{x:.2f}<br>Real: %{y:.2f}<br>%{text}',
))
fig_cal_all.add_trace(go.Scatter(
    x=cal_all_odds['pred_mean'], y=cal_all_odds['actual_mean'],
    mode='markers+lines', name='Odds',
    marker=dict(size=10, color=c_odds), line=dict(color=c_odds),
    text=[f'n={c}' for c in cal_all_odds['count']],
    hovertemplate='Previsto: %{x:.2f}<br>Real: %{y:.2f}<br>%{text}',
))

fig_cal_all.update_layout(
    title='Calibracao geral (H+D+A combinados): Modelo vs Odds',
    xaxis_title='Probabilidade prevista',
    yaxis_title='Frequencia real',
    template='plotly_white',
    height=500,
    width=600,
)
fig_cal_all.show()

## 7. Mapa de divergencias (top 20)

In [ ]:
# Calcular divergencia maxima entre modelo e odds por jogo
has_odds['div_home'] = has_odds['model_p_home'] - has_odds['odds_p_home']
has_odds['div_draw'] = has_odds['model_p_draw'] - has_odds['odds_p_draw']
has_odds['div_away'] = has_odds['model_p_away'] - has_odds['odds_p_away']
has_odds['max_div'] = has_odds[['div_home', 'div_draw', 'div_away']].abs().max(axis=1)

# Quem "acertou" — o resultado era a maior probabilidade?
def who_was_right(row):
    result = row['result']
    # Probabilidade atribuida ao resultado correto
    if result == 'H':
        p_model = row['model_p_home']
        p_odds = row['odds_p_home']
    elif result == 'D':
        p_model = row['model_p_draw']
        p_odds = row['odds_p_draw']
    else:
        p_model = row['model_p_away']
        p_odds = row['odds_p_away']
    
    if p_model > p_odds:
        return 'Modelo'
    elif p_odds > p_model:
        return 'Odds'
    return 'Empate'

has_odds['closer_to_result'] = has_odds.apply(who_was_right, axis=1)

# Top 20 divergencias
top_div = has_odds.nlargest(20, 'max_div')[[
    'home_name', 'away_name', 'season_year', 'game_week',
    'homeGoalCount', 'awayGoalCount', 'result',
    'model_p_home', 'odds_p_home',
    'model_p_draw', 'odds_p_draw',
    'model_p_away', 'odds_p_away',
    'max_div', 'closer_to_result',
]].copy()

top_div.columns = [
    'Mandante', 'Visitante', 'Ano', 'Rodada',
    'Gols_H', 'Gols_A', 'Res',
    'Mod_H', 'Odds_H',
    'Mod_D', 'Odds_D',
    'Mod_A', 'Odds_A',
    'Max_Div', 'Acertou',
]

print('=== Top 20 maiores divergencias (modelo vs odds) ===')
print(f'Modelo acertou mais: {(top_div["Acertou"] == "Modelo").sum()}/20')
print(f'Odds acertaram mais: {(top_div["Acertou"] == "Odds").sum()}/20')
top_div

In [ ]:
# Analise de padroes nas divergencias
print('=== Analise de padroes nas divergencias ===')

# 1. Em quantos jogos totais o modelo vs odds deu probabilidade maior ao resultado correto?
total_closer = has_odds['closer_to_result'].value_counts()
print(f'\nEm todos os {len(has_odds)} jogos:')
for k, v in total_closer.items():
    print(f'  {k}: {v} ({100*v/len(has_odds):.1f}%)')

# 2. O modelo diverge mais quando o mandante e favorito ou azarao?
has_odds['mandante_favorito_odds'] = has_odds['odds_p_home'] > 0.45
has_odds['mandante_azarao_odds'] = has_odds['odds_p_home'] < 0.30

print(f'\nDivergencia media quando mandante e favorito (odds P(H) > 45%): '
      f'{has_odds[has_odds["mandante_favorito_odds"]]["max_div"].mean():.4f}')
print(f'Divergencia media quando mandante e azarao (odds P(H) < 30%): '
      f'{has_odds[has_odds["mandante_azarao_odds"]]["max_div"].mean():.4f}')
print(f'Divergencia media geral: {has_odds["max_div"].mean():.4f}')

# 3. Modelo tende a superestimar ou subestimar o mandante?
bias_home = (has_odds['model_p_home'] - has_odds['odds_p_home']).mean()
bias_draw = (has_odds['model_p_draw'] - has_odds['odds_p_draw']).mean()
bias_away = (has_odds['model_p_away'] - has_odds['odds_p_away']).mean()
print(f'\nVies sistematico (modelo - odds):')
print(f'  P(H): {bias_home:+.4f} ({"modelo superestima mandante" if bias_home > 0 else "modelo subestima mandante"})')
print(f'  P(D): {bias_draw:+.4f} ({"modelo superestima empate" if bias_draw > 0 else "modelo subestima empate"})')
print(f'  P(A): {bias_away:+.4f} ({"modelo superestima visitante" if bias_away > 0 else "modelo subestima visitante"})')

In [ ]:
# Grafico: Distribuicao das divergencias
fig_div = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distribuicao de |divergencia max|', 'Modelo vs Odds: P(H) scatter'],
)

# Histograma de divergencias
fig_div.add_trace(
    go.Histogram(x=has_odds['max_div'], nbinsx=30, name='|Div max|',
                 marker_color='#636EFA', opacity=0.7),
    row=1, col=1,
)

# Scatter: P(H) modelo vs P(H) odds
colors = has_odds['result'].map({'H': '#00CC96', 'D': '#FFA15A', 'A': '#EF553B'})
fig_div.add_trace(
    go.Scatter(
        x=has_odds['odds_p_home'], y=has_odds['model_p_home'],
        mode='markers', name='P(H)',
        marker=dict(size=4, color=colors, opacity=0.5),
        text=has_odds.apply(
            lambda r: f"{r['home_name']} {int(r['homeGoalCount'])}x{int(r['awayGoalCount'])} {r['away_name']}",
            axis=1
        ),
        hovertemplate='%{text}<br>Odds P(H): %{x:.2f}<br>Modelo P(H): %{y:.2f}',
    ),
    row=1, col=2,
)
# Diagonal
fig_div.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
               line=dict(color='gray', dash='dash'), showlegend=False),
    row=1, col=2,
)

fig_div.update_xaxes(title_text='|Divergencia maxima|', row=1, col=1)
fig_div.update_yaxes(title_text='Frequencia', row=1, col=1)
fig_div.update_xaxes(title_text='Odds P(H)', row=1, col=2)
fig_div.update_yaxes(title_text='Modelo P(H)', row=1, col=2)

fig_div.update_layout(
    title='Divergencias entre Modelo e Odds',
    template='plotly_white',
    height=450,
    width=1100,
    showlegend=False,
)
fig_div.show()

In [ ]:
# Resumo final
print('=' * 60)
print('RESUMO — Fase 6: Analise de Odds')
print('=' * 60)
print(f'\nJogos analisados: {len(has_odds)}')
print(f'Temporadas: {sorted(has_odds["season_year"].unique())}')
print(f'\nBrier Score:')
print(f'  Modelo (DC+ELO): {m_model["brier_score"]:.5f}')
print(f'  Odds (casas):    {m_odds["brier_score"]:.5f}')
print(f'  Diferenca:       {m_model["brier_score"] - m_odds["brier_score"]:+.5f} '
      f'({"modelo pior" if m_model["brier_score"] > m_odds["brier_score"] else "modelo melhor"})')
print(f'\nLog-Loss:')
print(f'  Modelo (DC+ELO): {m_model["log_loss"]:.4f}')
print(f'  Odds (casas):    {m_odds["log_loss"]:.4f}')
print(f'  Diferenca:       {m_model["log_loss"] - m_odds["log_loss"]:+.4f} '
      f'({"modelo pior" if m_model["log_loss"] > m_odds["log_loss"] else "modelo melhor"})')
print(f'\nDivergencia media (modelo vs odds): {has_odds["max_div"].mean():.4f}')
print(f'Divergencia max: {has_odds["max_div"].max():.4f}')
print(f'\nVies sistematico:')
print(f'  P(H): {bias_home:+.4f}')
print(f'  P(D): {bias_draw:+.4f}')
print(f'  P(A): {bias_away:+.4f}')

## 8. Calibracao de parametros + Blend modelo-odds

Com base nas divergencias encontradas, re-calibramos `ELO_LAMBDA_WEIGHT` (de 0.15 para 0.10)
e adicionamos um blend com odds (`BLEND_ALPHA=0.34`) para reduzir o vies do mandante
e melhorar o Brier Score.

In [ ]:
# Re-importar com novos parametros
from src.config import BLEND_ALPHA
from src.poisson import blend_with_odds, odds_to_probs as odds_to_probs_fn

print(f'Parametros atualizados:')
print(f'  ELO_LAMBDA_WEIGHT = {ELO_LAMBDA_WEIGHT}')
print(f'  BLEND_ALPHA = {BLEND_ALPHA}')
print(f'  HFA = {HFA} (mantido)')

# Re-calcular probabilidades do modelo com ELO_LAMBDA_WEIGHT=0.10 (ja atualizado no config)
model_probs_new = []
for idx, match in has_odds.iterrows():
    home = match['home_name']
    away = match['away_name']
    lam_h, lam_a = calculate_lambdas(
        home, away, team_strengths, league_avgs,
        elo_ratings=elo_ratings,
    )
    probs = score_probabilities(lam_h, lam_a)
    model_probs_new.append({
        'idx': idx,
        'new_model_h': probs['home_win'],
        'new_model_d': probs['draw'],
        'new_model_a': probs['away_win'],
    })

new_df = pd.DataFrame(model_probs_new).set_index('idx')
has_odds['new_model_h'] = new_df['new_model_h']
has_odds['new_model_d'] = new_df['new_model_d']
has_odds['new_model_a'] = new_df['new_model_a']

# Calcular blend
blended = has_odds.apply(
    lambda r: blend_with_odds(
        r['new_model_h'], r['new_model_d'], r['new_model_a'],
        r['odds_p_home'], r['odds_p_draw'], r['odds_p_away'],
    ), axis=1, result_type='expand'
)
has_odds['blend_h'] = blended[0]
has_odds['blend_d'] = blended[1]
has_odds['blend_a'] = blended[2]

print(f'\nBlend calculado para {len(has_odds)} jogos')
print(f'Soma blend: min={has_odds[["blend_h","blend_d","blend_a"]].sum(axis=1).min():.6f}, '
      f'max={has_odds[["blend_h","blend_d","blend_a"]].sum(axis=1).max():.6f}')

In [ ]:
# Comparacao final: antes vs depois
m_new_model = compute_metrics(has_odds, 'new_model_h', 'new_model_d', 'new_model_a', 'Modelo novo (W=0.10)')
m_blend = compute_metrics(has_odds, 'blend_h', 'blend_d', 'blend_a', f'Blend (alpha={BLEND_ALPHA})')

print('=' * 65)
print('COMPARACAO FINAL: Modelo original vs Novo vs Blend vs Odds')
print('=' * 65)
for r in [m_model, m_new_model, m_blend, m_odds]:
    print(f'  {r["label"]:30s}  BS={r["brier_score"]:.5f}  LL={r["log_loss"]:.4f}')

# Vies de cada versao
for label, ph_col in [('Original (W=0.15)', 'model_p_home'),
                       ('Novo (W=0.10)', 'new_model_h'),
                       ('Blend', 'blend_h')]:
    bias = (has_odds[ph_col] - has_odds['odds_p_home']).mean()
    print(f'  Vies P(H) {label}: {bias:+.4f}')

print(f'\n  Melhoria BS (blend vs odds):     {m_odds["brier_score"] - m_blend["brier_score"]:+.5f}')
print(f'  Melhoria BS (blend vs original): {m_model["brier_score"] - m_blend["brier_score"]:+.5f}')
print(f'\n  O blend SUPERA as odds puras: '
      f'{"SIM" if m_blend["brier_score"] < m_odds["brier_score"] else "NAO"}')